# Data and splits

This notebook describes where the crops come from, how they were split, and the checks that keep the test sets clean. Every crop shows one building and has one grade from A to F (`docs/labelling_guideline.md`).

| Source | Crops | How the buildings were found | Labels |
|---|---|---|---|
| Original dataset | 1,050 from 361 photos | hand-drawn boxes | graded during an earlier study |
| Active-learning pool | 2,103 from 929 further photos | SAM 3 boxes, reviewed by hand (notebook 00) | graded in rounds of 100, then the rest in one batch |

Of the 2,103 pool crops, 1,185 were graded through active learning and bulk labelling (897 for training, 215 for validation, 73 left out as copies of test crops), and 345 graded crops make up tests 2 and 3. The rest failed the quality filter or were marked ungradable.

All photos are from the February 2023 earthquakes in Türkiye.

In [ ]:
# On Colab: install the package and get the configuration files.
!pip -q install "insarag-triage[roboflow] @ git+https://github.com/leeelyehezkel/insarag-damage-triage"
!git clone -q https://github.com/leeelyehezkel/insarag-damage-triage repo 2>/dev/null || true

## Download and assemble

The images are downloaded from the public Roboflow projects in `configs/data.yaml`, which needs a free Roboflow API key. `data/manifests/labels.csv` gives every crop's grade and split. Roboflow renames files on export, so crops are matched by an id built from the photo name, image number and box number.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from insarag_triage import core as L, analysis as A, active_learning as AL, data as DATA

REPO = Path('repo') if Path('repo').exists() else Path('.')
WORK = Path('work')
dcfg = DATA.load_config(REPO / 'configs' / 'data.yaml')
folders = {s: DATA.download(dcfg, s, WORK / 'roboflow') for s in ['original', 'al_labels', 'test12', 'test3']}
m = DATA.assemble(REPO / 'data' / 'manifests' / 'labels.csv', folders)
pd.crosstab(m.split, m.label, margins=True)

## The split by photo

A photo often shows several buildings. If crops from one photo end up in both training and test, the test measures memory of that scene rather than grading. The original 1,050 crops were therefore split by photo, stratified by grade, with 7 folds and seed 42: one fold for test, one for validation and five for training. That gave 723 / 169 / 158 crops, with no photo in two splits.

The code below repeats the procedure on the original crops. The fold assignment depends on the row order, which differs between Roboflow exports, so the exact crops in each split can differ from the study's. The manifest holds the split that was actually used.

In [ ]:
orig = m[m.subset == 'original'].reset_index(drop=True)
redo = L.make_splits(orig.assign(group=orig.photo), seed=42)
print('study split:', orig.split.replace({'test1': 'test'}).value_counts().to_dict())
print('redone split:', redo.split.value_counts().to_dict())
print('photos in more than one split (redone):', int(redo.groupby('group').split.nunique().gt(1).sum()))
L.split_table(redo)

## Image size: no stretching

The first export of the original crops resized every crop to 224 × 224 px, stretching it to a square. The crops from the pool kept their real shape. Training then mixed two different kinds of image, and fine detail such as cracks was lost. All results reported here use the un-stretched export, in which the models pad each crop to a square instead (letterbox). This cell checks for square crops.

In [ ]:
from PIL import Image
sizes = pd.DataFrame([Image.open(p).size for p in m.path], columns=['w', 'h'], index=m.index)
m['short_side'] = sizes.min(axis=1)
print(f"exactly square crops: {(sizes.w == sizes.h).mean():.1%} · median short side {int(m.short_side.median())} px")
m.groupby('subset').short_side.describe()[['count', '25%', '50%', '75%']]

## Quality filter

Pool crops were skipped when the short side was under 96 px, the aspect ratio was over 3, or the crop was very dark, very bright or nearly uniform. In a blind re-labelling, crops under 96 px were marked ungradable 43% of the time, against 5% of the others. The same thresholds are in `insarag_triage.pool.quality_ok`.

In [ ]:
from insarag_triage import pool as P
q = P.quality_table(m.assign(width=sizes.w, height=sizes.h))
m['quality_ok'] = P.quality_ok(q).to_numpy()
pd.crosstab(m.split, m.quality_ok)

## The three test sets

| Set | Crops | How it was made | Labels | Role |
|---|---|---|---|---|
| Test 1 | 158 | test fold of the original split | original labels, also re-labelled blind | comparison with the original dataset |
| Test 2 | 200 drawn, 193 gradeable | random pool crops, one per photo, from photos never labelled or picked | blind | tracking the active-learning rounds; choosing the model |
| Test 3 | 250 drawn, 154 gradeable, 152 used | as test 2, drawn before the rest of the pool was labelled | blind | final result, used once |

Blind means the crops were graded without any pre-filled label, prediction or tag, mixed together so that the set a crop came from couldn't be told. The photos of tests 2 and 3 were removed from the pool, so they could never be selected or trained on.

In [ ]:
for s in ['test1', 'test2', 'test3']:
    t = m[m.split == s]
    print(f"{s}: {len(t)} crops, {t.photo.nunique()} photos, per grade {t.label.value_counts().reindex(L.CLASSES, fill_value=0).to_dict()}")

## Leak check by image fingerprint

A split by photo name misses two things:

- **The same photo under two names.** The source data stores some photos twice, as `X` and `X__1_`.
- **A test crop picked into training.** In one notebook session the test-2 photos were not excluded from the pool, and active learning picked 30 of them.

Every crop is therefore compared by a 16 × 16 average-hash fingerprint. In the study this found 30 test-2 crops with copies in training and 2 duplicate photos in test 3; validation and test 1 were clean. The training copies are marked `excluded` in the manifest and left out of all final models. The comparison reads every image once and takes a few minutes.

In [ ]:
from insarag_triage.leakage import find_leaks
sets = {s: m.loc[m.split == s, 'path'].tolist() for s in ['train', 'val', 'test1', 'test2', 'test3']}
sets['excluded'] = m.loc[m.split == 'excluded', 'path'].tolist()     # the training copies removed after the check
leaks = find_leaks(sets, eval_sets=['val', 'test1', 'test2', 'test3'], train_sets=['train', 'excluded'])
print(f"identical pictures across training and evaluation: {leaks.fp.nunique()} groups")
print('by training set:', leaks[leaks.role == 'train'].set.value_counts().to_dict(), '(all should be in `excluded`)')
leaks.head(20)

## How consistent are the labels?

The 158 test-1 crops were graded a second time, blind, during this study (14 were marked ungradable). The two labellings agree exactly on 64% of crops and within one grade on 94% (Cohen's κ 0.56, quadratic-weighted κ 0.90). Agreement is lowest for A and B (0.42 and 0.41) and highest for C (0.91). The changes went in a direction: standing buildings were graded more severely the second time, collapsed ones less severely. This sets a rough ceiling for any model scored against these labels.

In [ ]:
from sklearn.metrics import cohen_kappa_score
t1 = m[(m.split == 'test1') & m.label_blind.isin(L.CLASSES)]
yo, yn = t1.label.map(L.CLASSES.index), t1.label_blind.map(L.CLASSES.index)
print(f"{len(t1)} crops · exact {np.mean(yo == yn):.3f} · within one grade {np.mean(abs(yo - yn) <= 1):.3f} · "
      f"kappa {cohen_kappa_score(yo, yn):.3f} · weighted kappa {cohen_kappa_score(yo, yn, weights='quadratic'):.3f}")
e = L.evaluate(yo, yn)
L.plot_confusion(e, f'Test 1: first label vs blind re-label (exact agreement {np.mean(yo == yn):.2f})', xlabel='blind re-label',
                 ylabel='first label', show_scores=False); plt.show()